Download metadata for each project, specifically querying for whatever is needed for batch correction

- right now, it resolves to the aliquot/sample-level (~10% of rows)
    - results in duplicates (potentially multiple rows per patient)
    - seen as duplicates per case_id
- later, just add this to the metadata extraction (DONE!)

## Setup

In [6]:
import sys
import json
import requests
import subprocess
import time
import datetime

import anndata as ad
import pandas as pd

from typing import Dict, List

from methyltrain.config.loader import load_config
from methyltrain.fs.layout import ProjectLayout
from methyltrain.constants import GDC_QUERY_URL, MAX_RETRIES, GDC_QUERY_BATCH_URL
from methyltrain.utils.utils import (
    verify_gdc_client,
    extract_project_id,
    extract_sample_type,
    extract_submitter_id,
    extract_aliquot_id
)
from methyltrain.utils.load_utils import load_audit_table, load_metadata
from methyltrain.utils.load_utils import load_status_log

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Fix each project

In [2]:
project_name = "UVM"

In [7]:
layout = ProjectLayout(
    project_name = f"TCGA-{project_name}",
    root_dir = f"../data",
    raw_dir = f"../data/raw/TCGA-{project_name}",
    audit_table = f"../data/metadata/TCGA-{project_name}/TCGA-{project_name}_audit_table.csv",
    metadata = f"../data/metadata/TCGA-{project_name}/TCGA-{project_name}_metadata.csv",
    manifest = f"../data/metadata/TCGA-{project_name}/TCGA-{project_name}_manifest.csv",
    status_log = f"../data/metadata/TCGA-{project_name}/TCGA-{project_name}_status_log.csv",
    project_adata = f"../data/processed/TCGA-{project_name}_adata.h5ad"
)

config = load_config("../config/local_config_hpc.yaml")
audit_table = load_audit_table(layout)
metadata = load_metadata(layout)
adata = ad.read_h5ad(f"../data/processed/TCGA-{project_name}_adata.h5ad")

In [12]:
biospecimen = build_biospecimen(metadata, config)

In [22]:
metadata = metadata.merge(biospecimen[['aliquot_id', 'barcode']], on = 'aliquot_id', how = 'left')
metadata['batch_id'] = metadata['barcode'].apply(extract_batch_id)
metadata

,file_name,data_type,data_category,experimental_strategy,platform,project_id,submitter_id,sample_type,aliquot_id,status,barcode,batch_id
0,1a031b65-b765-409f-816c-0a080ba52f61.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-YZ-A983,Primary Tumor,4f69d846-4c09-40f2-8061-2e0038557c0e,success,TCGA-YZ-A983-01A-11D-A39X-05,01A-A39X
1,a7cb10da-eda1-4804-92c1-17eb4f66d535.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-V4-A9ET,Primary Tumor,bb855206-a3c7-45ca-89b7-4b4a102c99e4,success,TCGA-V4-A9ET-01A-11D-A39X-05,01A-A39X
2,fb3f222e-052b-4d6e-b1e5-105508469273.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-WC-A880,Primary Tumor,fec3f5a3-b678-4d4e-b706-f339675fa164,success,TCGA-WC-A880-01A-11D-A39X-05,01A-A39X
3,9d6f989b-a7db-4947-94b5-c323d7de2077.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-V4-A9EW,Primary Tumor,fd0cbc07-0a93-4eed-86fb-56837f51fbb3,success,TCGA-V4-A9EW-01A-11D-A39X-05,01A-A39X
4,9c5fb413-6478-44c7-8e56-31f390556400.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-WC-A87T,Primary Tumor,6695c730-7799-4733-86fd-4b0035414d33,success,TCGA-WC-A87T-01A-11D-A39X-05,01A-A39X
...,...,...,...,...,...,...,...,...,...,...,...,...
72,96af0a46-0cf1-4510-8f33-af87b3694c17.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-V4-A9F3,Primary Tumor,f00c1703-43cc-4407-95a8-d1f2263e681f,success,TCGA-V4-A9F3-01A-11D-A39X-05,01A-A39X
73,7f79aa44-9106-4bbe-add8-25349982b730.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-YZ-A985,Primary Tumor,aff3c704-fac5-41ef-8f4c-d10ffc8e5e6c,success,TCGA-YZ-A985-01A-11D-A39X-05,01A-A39X
74,a0bc9249-921f-4ccc-acfe-2e4570713819.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-V4-A9EE,Primary Tumor,48e85d0e-2aed-4e23-87cf-d18213108c77,success,TCGA-V4-A9EE-01A-11D-A39X-05,01A-A39X
75,a5f2f159-1bdf-473c-95cb-e27af9aa4a7a.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UVM,TCGA-V4-A9EC,Primary Tumor,5420f8d2-342a-4b7d-9265-0eb99fdd57b6,success,TCGA-V4-A9EC-01A-11D-A39X-05,01A-A39X


In [223]:
plate_metadata = query_plate_metadata(adata)

# Preserve the file_id index
adata.obs = adata.obs.reset_index()
adata.obs = adata.obs.merge(plate_metadata[['aliquot_id', 'barcode']], on = 'aliquot_id', how = 'left')
adata.obs = adata.obs.set_index('file_id')

adata.obs['batch_id'] = adata.obs['barcode'].apply(extract_batch_id)
adata.write_h5ad(f"../data/processed/TCGA-{project_name}_adata.h5ad")

/home/campbell/sophiali/miniforge3/envs/methyltrain-env/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/campbell/sophiali/miniforge3/envs/methyltrain-env/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


## Helper Functions

In [11]:
def build_biospecimen(metadata: pd.DataFrame,
                      config: Dict, 
                      batch_size: int = 20) -> pd.DataFrame:
    """
    Queries for sample biospecimen metadata used for batch correction
    """

    # Fetch the case IDs of all samples successfully downloaded
    case_ids = metadata['submitter_id'].unique().tolist()
    all_hits = []

    # Query the GDC client for the biospecimen data
    for i in range(0, len(case_ids), batch_size):
        batch = case_ids[i:i + batch_size]

        filters = {
            "op": "in",
            "content": {"field": "submitter_id", "value": batch}
        }
    
        params = {
            "filters": json.dumps(filters),
            "expand": "samples.portions.analytes.aliquots",
            "fields": ",".join(['submitter_id', 'samples.portions.submitter_id',
                                'samples.portions.analytes.aliquots.aliquot_id',
                                'samples.portions.analytes.aliquots.submitter_id']),
            "format": "JSON",
            "size": len(batch)
        }

        # Query the `cases` endpoint
        response = requests.get(GDC_QUERY_BATCH_URL, params = params)
        response.raise_for_status()
        hits = response.json()['data']['hits']

        # Extract the nested barcode values
        for case in hits:
            case_id = case['submitter_id']
            for sample in case.get('samples', []):
                for portion in sample.get('portions', []):
                    portion_barcode = portion.get('submitter_id')
                    for analyte in portion.get('analytes', []):
                        for aliquot in analyte.get('aliquots', []):

                            # Use the aliquot-level barcode if available, else portion-level
                            aliquot_barcode = aliquot.get('submitter_id', portion_barcode)
                            all_hits.append({
                                "case_id": case_id,
                                "aliquot_id": aliquot.get('aliquot_id'),
                                "barcode": aliquot_barcode
                            })

    # Keep only rows corresponding to aliquots in adata
    biospecimen = pd.DataFrame(all_hits)
    biospecimen = biospecimen[biospecimen['aliquot_id'].isin(metadata['aliquot_id'])]

    # Drop exact duplicates
    biospecimen = biospecimen.drop_duplicates(subset = 'aliquot_id')
    biospecimen['aliquot_id'] = biospecimen['aliquot_id'].astype(str)
    
    return biospecimen

In [157]:
def query_plate_metadata(adata):
    # Query for samples.submitter_id for the full TCGA barcode
    case_ids = adata.obs['submitter_id'].unique().tolist()
    all_hits = []
    
    chunk_size = 25
    
    for i in range(0, len(case_ids), chunk_size):
        batch = case_ids[i:i + chunk_size]
        filters = {
            "op": "in",
            "content": {"field": "submitter_id", "value": batch}
        }
    
        params = {
            "filters": json.dumps(filters),
            "expand": "samples.portions.analytes.aliquots",
            "fields": "submitter_id,samples.portions.submitter_id,samples.portions.analytes.aliquots.aliquot_id,samples.portions.analytes.aliquots.submitter_id",
            "format": "JSON",
            "size": len(batch)
        }
    
        response = requests.get(GDC_QUERY_BATCH_URL, params=params)
        response.raise_for_status()
        hits = response.json()['data']['hits']
    
        for case in hits:
            case_id = case['submitter_id']
            for sample in case.get('samples', []):
                for portion in sample.get('portions', []):
                    portion_barcode = portion.get('submitter_id')  # full portion-level barcode
                    for analyte in portion.get('analytes', []):
                        for aliquot in analyte.get('aliquots', []):

                            # Use the aliquot-level barcode if available, else portion-level
                            aliquot_barcode = aliquot.get('submitter_id', portion_barcode)
                            all_hits.append({
                                "case_id": case_id,
                                "aliquot_id": aliquot.get('aliquot_id'),
                                "barcode": aliquot_barcode
                            })
    
    # Keep only rows corresponding to aliquots in adata
    plate_metadata = pd.DataFrame(all_hits)
    plate_metadata = plate_metadata[plate_metadata['aliquot_id'].isin(adata.obs['aliquot_id'])]

    # Drop exact duplicates
    plate_metadata = plate_metadata.drop_duplicates(subset = 'aliquot_id')
    plate_metadata['aliquot_id'] = plate_metadata['aliquot_id'].astype(str)
    
    return plate_metadata

In [18]:
def extract_batch_id(barcode: str):
    # Returns the portion and the plate ID together
    if pd.isna(barcode): return None
    parts = barcode.split('-')
    if len(parts) >= 6:
        return f"{parts[3]}-{parts[5]}"  # portion + plate
    return None